# II. Bài tập hướng dẫn mẫu
b. Thuật toán tìm kiếm A* trên đồ thị:

In [18]:
from collections import deque

class Graph:
    def __init__(self, adjac_lis):
        self.adjac_lis = adjac_lis

    def get_neighbors(self, v):
        return self.adjac_lis[v]

    def h(self, n):
        H = {
            'A': 1,
            'B': 1,
            'C': 1,
            'D': 1
        }
        return H[n]

    def a_star_algorithm(self, start, stop):
        open_lst = set([start])
        closed_lst = set([])

        poo = {}
        poo[start] = 0

        par = {}
        par[start] = start

        while len(open_lst) > 0:
            n = None

            for v in open_lst:
                if n == None or poo[v] + self.h(v) < poo[n] + self.h(n):
                    n = v

            if n == None:
                print('Path does not exist!')
                return None

            if n == stop:
                reconst_path = []

                while par[n] != n:
                    reconst_path.append(n)
                    n = par[n]

                reconst_path.append(start)
                reconst_path.reverse()

                print('Path found: {}'.format(reconst_path))
                return reconst_path

            for (m, weight) in self.get_neighbors(n):
                if m not in open_lst and m not in closed_lst:
                    open_lst.add(m)
                    par[m] = n
                    poo[m] = poo[n] + weight

                else:
                    if poo[m] > poo[n] + weight:
                        poo[m] = poo[n] + weight
                        par[m] = n

                        if m in closed_lst:
                            closed_lst.remove(m)
                            open_lst.add(m)

            open_lst.remove(n)
            closed_lst.add(n)

        print('Path does not exist!')
        return None

adjac_lis = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)],
}

graph1 = Graph(adjac_lis)
graph1.a_star_algorithm('A', 'C')

Path found: ['A', 'B', 'C']


['A', 'B', 'C']

# III. Bài tập ở lớp
Bài 2. Cài đặt thuật giải A* tìm đường đi giữa 2 đỉnh bất kỳ trong đồ thị.

In [59]:
import heapq
import math

def main():

    graph = {
        'A': {'B': 2, 'C': 5},
        'B': {'A': 2, 'D': 3, 'E': 6},
        'C': {'A': 5, 'F': 4},
        'D': {'B': 3, 'E': 1, 'G': 7},
        'E': {'B': 6, 'D': 1, 'H': 8},
        'F': {'C': 4, 'I': 3},
        'G': {'D': 7, 'H': 2},
        'H': {'E': 8, 'G': 2, 'J': 5},
        'I': {'F': 3, 'J': 4},
        'J': {'H': 5, 'I': 4}
    }

    coords = {
        'A': (0, 0), 'B': (2, 1), 'C': (5, 0), 'D': (3, 3), 'E': (4, 4),
        'F': (8, 1), 'G': (5, 5), 'H': (6, 6), 'I': (10, 2), 'J': (8, 7)
    }

    start = 'A'
    goal = 'J'

    def heuristic(node):
        x1, y1 = coords[node]
        x2, y2 = coords[goal]
        return math.hypot(x1 - x2, y1 - y2)

    open_set = []
    heapq.heappush(open_set, (heuristic(start), 0, start, None))
    g_score = {start: 0}
    parent = {start: None}
    closed_set = set()

    print(f"Tìm đường từ {start} đến {goal} ===")

    while open_set:
        f, g, node, par = heapq.heappop(open_set)
        if node in closed_set:
            continue
        closed_set.add(node)
        if par is not None:
            parent[node] = par

        if node == goal:

            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            print(f"Đường đi ngắn nhất: {' -> '.join(path)}")
            print(f"Tổng chi phí: {g}")
            return

        for neighbor, weight in graph[node].items():
            if neighbor in closed_set:
                continue
            tentative_g = g + weight
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                f_score = tentative_g + heuristic(neighbor)
                heapq.heappush(open_set, (f_score, tentative_g, neighbor, node))

    print(f"Không tìm thấy đường đi từ {start} đến {goal}")

if __name__ == "__main__":
    main()

Tìm đường từ A đến J ===
Đường đi ngắn nhất: A -> C -> F -> I -> J
Tổng chi phí: 16


# IV. Bài tập về nhà
Bài 2. Cài đặt bài toán người giao hàng theo thuật giải A*.

In [60]:
import heapq

def main():

    dist = [
        [0, 10, 15, 20, 25],
        [10, 0, 35, 25, 30],
        [15, 35, 0, 30, 5],
        [20, 25, 30, 0, 15],
        [25, 30, 5, 15, 0]
    ]
    n = len(dist)
    start_city = 0

    def heuristic(visited_mask, current):
        unvisited = [i for i in range(n) if not (visited_mask >> i) & 1]
        if not unvisited:

            return dist[current][start_city]
        if len(unvisited) == 1:
            u = unvisited[0]
            return dist[current][u] + dist[u][start_city]

        m = len(unvisited)

        node_idx = {node: idx for idx, node in enumerate(unvisited)}

        sub_dist = [[dist[unvisited[i]][unvisited[j]] for j in range(m)] for i in range(m)]

        in_mst = [False] * m
        min_edge = [float('inf')] * m
        min_edge[0] = 0
        mst_cost = 0
        for _ in range(m):
            u = -1
            best = float('inf')
            for i in range(m):
                if not in_mst[i] and min_edge[i] < best:
                    best = min_edge[i]
                    u = i
            if u == -1:
                break
            in_mst[u] = True
            mst_cost += best
            for v in range(m):
                if not in_mst[v] and sub_dist[u][v] < min_edge[v]:
                    min_edge[v] = sub_dist[u][v]

        min_from_current = min(dist[current][v] for v in unvisited)

        min_to_start = min(dist[v][start_city] for v in unvisited)
        return mst_cost + min_from_current + min_to_start

    start_mask = 1 << start_city
    start_state = (start_city, start_mask)
    goal_mask = (1 << n) - 1

    open_set = []
    heapq.heappush(open_set, (heuristic(start_mask, start_city), 0, start_city, start_mask, None))
    g_score = {start_state: 0}
    parent = {start_state: (None, None)}
    closed_set = set()

    while open_set:
        f, g, cur, mask, par = heapq.heappop(open_set)
        state = (cur, mask)
        if state in closed_set:
            continue
        closed_set.add(state)
        if par is not None:
            parent[state] = par

        if mask == goal_mask:

            total_cost = g + dist[cur][start_city]

            path = []
            s = state
            while s[0] is not None:
                path.append(s[0])
                s = parent.get(s, (None, None))
                if s is None:
                    break
            path.reverse()
            path.append(start_city)
            print(f"Tìm thấy tour tối ưu với tổng chi phí: {total_cost}")
            print("Thứ tự các thành phố:", " -> ".join(map(str, path)))
            return

        for nxt in range(n):
            if (mask >> nxt) & 1:
                continue
            new_mask = mask | (1 << nxt)
            tentative_g = g + dist[cur][nxt]
            new_state = (nxt, new_mask)
            if new_state not in g_score or tentative_g < g_score[new_state]:
                g_score[new_state] = tentative_g
                h = heuristic(new_mask, nxt)
                heapq.heappush(open_set, (tentative_g + h, tentative_g, nxt, new_mask, state))

    print("Không tìm thấy lời giải TSP.")

if __name__ == "__main__":
    main()


Tìm thấy tour tối ưu với tổng chi phí: 70
Thứ tự các thành phố: 0 -> 1 -> 3 -> 4 -> 2 -> 0
